In [15]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import numpy as np
import scipy
import copy

from scipy.sparse import coo_matrix, block_diag, identity, hstack, csr_matrix, csc_matrix, vstack
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import time 
import matplotlib as mpl

from pyiga import assemble, bspline, vform, geometry, vis, solvers, utils, topology, ieti, algebra, operators, adaptive
from pyiga import algebra_cy, ieti_cy, bspline_cy

from scipy.sparse.linalg import aslinearoperator as LinOp

np.set_printoptions(linewidth=100000)
np.set_printoptions(precision=5)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
u = lambda x,y,z: np.sin(x)*np.cos(y)*np.sin(z)

In [21]:
deg,N=5,10
geo = geometry.twisted_box()
kvs = 3*(bspline.make_knots(deg,0,1,N),)
kvs0 = 3*(bspline.make_knots(0,0,1,N),)

In [22]:
Mh = assemble.assemble('u * v * dx', kvs=kvs, geo=geo)
Uh = assemble.assemble('f * v * dx', kvs=kvs, geo=geo,f=u).ravel()

In [23]:
uh_ = operators.make_solver(Mh, spd=True)@Uh
uh = bspline.BSplineFunc(kvs,uh_)

In [26]:
assemble.assemble('(u-0*uh)**2*v*dx', kvs=kvs0, geo=geo, u=u, uh=uh, quadorder=deg+1).sum()

0.27181284455047217

In [27]:
assemble.integrate(kvs0, f = lambda x,y,z: (u(x,y,z))**2,geo=geo,f_physical=True, nqp = deg+1)

0.2718128445504721

NameError: name 'M' is not defined

In [18]:
uh.coeffs

array([[[ 1.14111e-07, -5.68679e-05, -1.64212e-04, -2.53505e-04, -2.84305e-04],
        [ 1.14106e-07, -5.68654e-05, -1.64204e-04, -2.53493e-04, -2.84292e-04],
        [ 1.01542e-07, -5.06043e-05, -1.46125e-04, -2.25583e-04, -2.52991e-04],
        [ 7.78168e-08, -3.87806e-05, -1.11983e-04, -1.72875e-04, -1.93879e-04],
        [ 6.16218e-08, -3.07097e-05, -8.86773e-05, -1.36897e-04, -1.53530e-04]],

       [[-5.68679e-05,  2.83406e-02,  8.18362e-02,  1.26336e-01,  1.41685e-01],
        [-5.68654e-05,  2.83393e-02,  8.18325e-02,  1.26330e-01,  1.41679e-01],
        [-5.06043e-05,  2.52190e-02,  7.28225e-02,  1.12421e-01,  1.26080e-01],
        [-3.87806e-05,  1.93266e-02,  5.58074e-02,  8.61536e-02,  9.66211e-02],
        [-3.07097e-05,  1.53044e-02,  4.41930e-02,  6.82237e-02,  7.65127e-02]],

       [[-1.64212e-04,  8.18362e-02,  2.36310e-01,  3.64808e-01,  4.09131e-01],
        [-1.64204e-04,  8.18325e-02,  2.36299e-01,  3.64791e-01,  4.09113e-01],
        [-1.46125e-04,  7.28225e-02,